# 3.3 Pandas'ta İşlemler

Bu notebook, PDS Handbook (TR) web sayfasının **Türkçe Jupyter karşılığıdır** — aynı açıklamalar, ders notları ve kod örnekleri.

| | |
|---|---|
| **Web sayfası** | `chapters/03-pandas/03-operations-in-pandas.html` |
| **Çalıştırma** | JupyterLab, VS Code veya Colab — hücreleri **yukarıdan aşağı** sırayla (`Shift+Enter`) |
| **Bağımlılık** | Kod hücreleri birbirine bağlıdır; hata alırsanız önce üsttekileri çalıştırın |

> **Kaynak:** Jake VanderPlas, *Python Data Science Handbook* — Türkçe ders uyarlaması



Orijinal: Operating on Data in Pandas

NumPy'nin güçlü yanlarından biri, temel aritmetik (toplama, çıkarma, çarpma vb.) ve daha karmaşık işlemler (trigonometrik, üstel ve logaritmik fonksiyonlar vb.) dahil hızlı eleman bazlı işlemler yapmamıza olanak tanımasıdır. Pandas bu işlevselliğin büyük kısmını NumPy'den devralır; 2.3 Evrensel Fonksiyonlar bölümünde tanıtılan ufunc'lar bunun temelidir.

Pandas birkaç kullanışlı ekleme sunar: tek terimli işlemlerde (negasyon, trigonometrik fonksiyonlar vb.) ufunc'lar çıktıda indeks ve sütun etiketlerini korur; ikili işlemlerde (toplama, çarpma vb.) Pandas nesneleri ufunc'e aktarılırken indeksleri otomatik hizalar. Ham NumPy dizilerinde hata kaynağı olabilen veri bağlamını koruma ve farklı kaynaklardan veri birleştirme, Pandas ile neredeyse hatasız hale gelir. Ayrıca bir boyutlu Series ile iki boyutlu DataFrame arasında iyi tanımlanmış işlemler vardır.

## Ufunc'lar: İndeks Korunması

Pandas NumPy ile çalışacak şekilde tasarlandığından, herhangi bir NumPy ufunc'u Pandas Series ve DataFrame nesnelerinde çalışır. Bunu göstermek için basit bir Series ve DataFrame tanımlayalım:


In [ ]:
# import_pd_np.py
import pandas as pd
import numpy as np



In [ ]:
# ser_ornek.py
rng = np.random.default_rng(42)
ser = pd.Series(rng.integers(0, 10, 4))
ser



In [ ]:
# df_ornek.py
df = pd.DataFrame(rng.integers(0, 10, (3, 4)),
                  columns=['A', 'B', 'C', 'D'])
df



Bu nesnelerden birine NumPy ufunc uygularsak sonuç, indeksleri koruyan başka bir Pandas nesnesidir:


In [ ]:
# np_exp_ser.py
np.exp(ser)



Daha karmaşık işlem dizileri için de geçerlidir:


In [ ]:
# np_sin_df.py
np.sin(df * np.pi / 4)



2.3 Evrensel Fonksiyonlar bölümündeki ufunc'ların hepsi benzer şekilde kullanılabilir.

> **Not**
>

## Ufunc'lar: İndeks Hizalama

İki Series veya DataFrame üzerinde ikili işlemlerde Pandas, işlem sırasında indeksleri hizalar. Eksik veriyle çalışırken bu çok kullanışlıdır.

### Series'te İndeks Hizalama

Örnek: iki farklı kaynaktan veri birleştirip ABD'nin alan ve nüfus açısından ilk üç eyaletini bulmak isteyelim:


In [ ]:
# area_pop.py
area = pd.Series({'Alaska': 1723337, 'Texas': 695662,
                  'California': 423967}, name='area')
population = pd.Series({'California': 39538223, 'Texas': 29145505,
                        'Florida': 21538187}, name='population')



Nüfus yoğunluğu için bunları bölelim:


In [ ]:
# pop_density.py
population / area



Sonuç dizisi, iki girdi dizisinin indekslerinin birleşimini içerir; doğrudan şöyle de bulunabilir:


In [ ]:
# index_union.py
area.index.union(population.index)



Birinde veya diğerinde giriş olmayan her öğe NaN (Not a Number) ile işaretlenir — Pandas'ın eksik veri göstergesidir (3.4 Eksik Veri). Bu indeks eşleştirmesi Python'un yerleşik aritmetik ifadelerinin tümünde böyledir; eksik değerler NaN ile işaretlenir:


In [ ]:
# series_align_nan.py
A = pd.Series([2, 4, 6], index=[0, 1, 2])
B = pd.Series([1, 3, 5], index=[1, 2, 3])
A + B



> **Not**
>

NaN istenmiyorsa, operatörler yerine nesne yöntemleriyle fill_value değiştirilebilir. A.add(B), A + B ile eşdeğerdir ancak eksik öğeler için doldurma değeri belirtilebilir:


In [ ]:
# add_fill_value.py
A.add(B, fill_value=0)



### DataFrame'de İndeks Hizalama

DataFrame işlemlerinde hem sütunlar hem indeksler için benzer hizalama yapılır:


In [ ]:
# df_A.py
A = pd.DataFrame(rng.integers(0, 20, (2, 2)),
                 columns=['a', 'b'])
A



In [ ]:
# df_B.py
B = pd.DataFrame(rng.integers(0, 10, (3, 3)),
                 columns=['b', 'a', 'c'])
B



In [ ]:
# df_A_plus_B.py
A + B



İndeksler iki nesnedeki sıradan bağımsız doğru hizalanır; sonuçtaki indeksler sıralanır. Series'te olduğu gibi aritmetik yöntemler ve fill_value kullanılabilir. Burada A'daki tüm değerlerin ortalamasıyla dolduruyoruz:


In [ ]:
# df_add_fill.py
A.add(B, fill_value=A.values.mean())



Python operatörleri ve eşdeğer Pandas yöntemleri:

## Ufunc'lar: DataFrame ve Series Arası İşlemler

DataFrame ile Series arasında işlem yaparken indeks ve sütun hizalaması korunur; sonuç, iki boyutlu ile tek boyutlu NumPy dizisi işlemine benzer. Yaygın örnek: iki boyutlu diziden bir satırını çıkarmak:


In [ ]:
# matris_A.py
A = rng.integers(10, size=(3, 4))
A



In [ ]:
# numpy_broadcast_row.py
A - A[0]



NumPy broadcasting kurallarına göre iki boyutlu dizi ile bir satırı arasındaki çıkarma satır bazında uygulanır. Pandas'ta da varsayılan olarak satır bazında çalışır:


In [ ]:
# df_minus_row.py
df = pd.DataFrame(A, columns=['Q', 'R', 'S', 'T'])
df - df.iloc[0]



Sütun bazında çalışmak için daha önceki nesne yöntemlerini axis ile kullanın:


In [ ]:
# df_subtract_col.py
df.subtract(df['R'], axis=0)



Bu DataFrame/Series işlemleri de önceki gibi iki öğe arasında indeksleri otomatik hizalar:


In [ ]:
# halfrow.py
halfrow = df.iloc[0, ::2]
halfrow



In [ ]:
# df_minus_halfrow.py
df - halfrow



İndeks ve sütunların korunması ve hizalanması, Pandas işlemlerinin veri bağlamını sürdürmesini sağlar; heterojen veya hizasız ham NumPy dizilerinde sık görülen hatalar önlenir.

### 🧪 Şimdi deneyin

🧪 Şimdi deneyin
      İki Series oluşturup farklı indekslerle toplayın; fill_value=0 ile sonucu karşılaştırın:
          
      import pandas as pd
A = pd.Series([10, 20], index=['x', 'y'])
B = pd.Series([1, 2, 3], index=['y', 'z', 'w'])
print("A + B:\n", A + B)
print("\nA.add(B, fill_value=0):\n", A.add(B, fill_value=0))

### 🧪 Şimdi deneyin

🧪 Şimdi deneyin
      Küçük bir DataFrame oluşturup ilk satırı çıkarın (df - df.iloc[0]):
          
      import pandas as pd
import numpy as np
rng = np.random.default_rng(0)
df = pd.DataFrame(rng.integers(0, 10, (3, 4)), columns=list('ABCD'))
print(df)
print("\nSatır farkı:\n", df - df.iloc[0])

> **Not**
>

> **Not**
>

> **Not**
>
